In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from itertools import product
import copy
import sys

!rm -r /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

rm: cannot remove '/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory
Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 200, done.
remote: Counting objects: 100% (200/200), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 200 (delta 106), reused 191 (delta 98), pack-reused 0 (from 0)
Receiving objects: 100% (200/200), 1.95 MiB | 32.27 MiB/s, done.
Resolving deltas: 100% (106/106), done.
8524bd2 (HEAD -> main, origin/main, origin/HEAD) typo


### Import necessary functions/modules from git repository

In [2]:
sys.path.append("/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection")
from dataset import LSTM_Dataset
from utils import load_config
import lstm
import train
from tqdm.notebook import tqdm
if torch.cuda.is_available():
    device = torch.device("cuda")
base_config = load_config("/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection/config.yaml")

### load device, datasets, config into environment.

In [3]:
train_dataset = LSTM_Dataset(device, 100)
test_dataset = LSTM_Dataset(device, 100, start = 90000, end = 100000)

In [4]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### First hyperparam sweep

In [5]:
lr_list = [5e-4]
hidden_sizes = [128, 256]
batch_sizes = [32]
training_results = {}
configs = [
    (lr, bs, hs)
    for lr, bs, hs in product(lr_list, batch_sizes, hidden_sizes)
]
for lr, bs, hs in tqdm(configs, desc="Sweep"):
    config = copy.deepcopy(base_config)
    
    config["training"]["lr"] = lr
    config["training"]["batch_size"] = bs
    config["training"]["num_epochs"] = 100
    config["model"]["hidden_size"] = hs
    
    exp_name = f"lr{lr}_bs{bs}_hs{hs}"
    model = lstm.LSTMBaseline(**config["model"]).to(device)
    weights, train_arr, test_arr = train.fit_lstm(
        model, exp_name, train_dataset, test_dataset, **config["training"]
    )
    training_results[exp_name] = {
        "hyperparams": {"lr": lr, "bs": bs, "hs": hs},
        "weights": weights,
        "train": train_arr,
        "test": test_arr,
        "best_test": min(test_arr)
    }
    print()

best_exp = min(training_results, key=lambda k: training_results[k]["best_test"])
best_hs  = training_results[best_exp]["hyperparams"]["hs"]
print(f"Best config: {best_exp}, hs={best_hs}")

Sweep:   0%|          | 0/2 [00:00<?, ?it/s]

|lr0.0005_bs32_hs128| train = 0.0188, test= 0.0595
update LR: 0.0005 -> 0.00025
|lr0.0005_bs32_hs128| train = 0.0122, test= 0.0384
update LR: 0.00025 -> 0.000125
|lr0.0005_bs32_hs128| train = 0.0110, test= 0.0334
|lr0.0005_bs32_hs128| train = 0.0109, test= 0.0319
|lr0.0005_bs32_hs128| train = 0.0108, test= 0.0313
|lr0.0005_bs32_hs128| train = 0.0108, test= 0.0305
update LR: 0.000125 -> 6.25e-05
|lr0.0005_bs32_hs128| train = 0.0105, test= 0.0273
update LR: 6.25e-05 -> 3.125e-05
|lr0.0005_bs32_hs128| train = 0.0104, test= 0.0267
update LR: 3.125e-05 -> 1.5625e-05
update LR: 1.5625e-05 -> 7.8125e-06
update LR: 7.8125e-06 -> 3.90625e-06
|lr0.0005_bs32_hs128| train = 0.0103, test= 0.0265
update LR: 3.90625e-06 -> 1.953125e-06
update LR: 1.953125e-06 -> 9.765625e-07
|lr0.0005_bs32_hs128| train = 0.0103, test= 0.0264

|lr0.0005_bs32_hs256| train = 0.0195, test= 0.0478
update LR: 0.0005 -> 0.00025
|lr0.0005_bs32_hs256| train = 0.0119, test= 0.0273
update LR: 0.00025 -> 0.000125
|lr0.0005_bs32_

### Second hyperparam sweep

In [6]:
lr_list = [1e-3, 5e-4, 2.5e-4, 1e-4]
hidden_sizes = [best_hs]
batch_sizes  = [32]
second_sweep_training_results = dict()
second_sweep_configs = [
    (lr, bs, hs)
    for lr, bs, hs in product(lr_list, batch_sizes, hidden_sizes)
]
for lr, bs, hs in tqdm(second_sweep_configs, desc="Sweep"):
    config = copy.deepcopy(base_config)
    
    config["training"]["lr"] = lr
    config["training"]["batch_size"] = bs
    config["training"]["num_epochs"] = 100
    config["model"]["hidden_size"] = hs
    
    exp_name = f"lr{lr}_bs{bs}_hs{hs}"
    model = lstm.LSTMBaseline(**config["model"]).to(device)
    weights, train_arr, test_arr = train.fit_lstm(
        model, exp_name, train_dataset, test_dataset, **config["training"]
    )
    second_sweep_training_results[exp_name] = {
        "hyperparams": {"lr": lr, "bs": bs, "hs": hs},
        "weights": weights,
        "train": train_arr,
        "test": test_arr,
        "best_test": min(test_arr)
    }
    print()

best_second_exp = min(second_sweep_training_results, key=lambda k: second_sweep_training_results[k]["best_test"])
best_lr  = second_sweep_training_results[best_second_exp]["hyperparams"]["lr"]
print(f"Best config: {best_second_exp}, hlr={best_lr}")

Sweep:   0%|          | 0/4 [00:00<?, ?it/s]

|lr0.001_bs32_hs256| train = 0.0219, test= 0.0609
update LR: 0.001 -> 0.0005
|lr0.001_bs32_hs256| train = 0.0127, test= 0.0368
update LR: 0.0005 -> 0.00025
|lr0.001_bs32_hs256| train = 0.0115, test= 0.0325
update LR: 0.00025 -> 0.000125
update LR: 0.000125 -> 6.25e-05
|lr0.001_bs32_hs256| train = 0.0103, test= 0.0293
update LR: 6.25e-05 -> 3.125e-05
update LR: 3.125e-05 -> 1.5625e-05
| experiment: lr0.001_bs32_hs256 | epoch 44, train: MSE 0.0101, test MSE: 0.0317
Stopping early

|lr0.0005_bs32_hs256| train = 0.0189, test= 0.0374
|lr0.0005_bs32_hs256| train = 0.0159, test= 0.0343
update LR: 0.0005 -> 0.00025
update LR: 0.00025 -> 0.000125
|lr0.0005_bs32_hs256| train = 0.0112, test= 0.0247
update LR: 0.000125 -> 6.25e-05
update LR: 6.25e-05 -> 3.125e-05
|lr0.0005_bs32_hs256| train = 0.0104, test= 0.0279
update LR: 3.125e-05 -> 1.5625e-05
update LR: 1.5625e-05 -> 7.8125e-06
| experiment: lr0.0005_bs32_hs256 | epoch 46, train: MSE 0.0102, test MSE: 0.0265
Stopping early

|lr0.00025_bs32_hs

### Third Hyperparam sweep

In [7]:
lr_list = [best_lr]
hidden_sizes = [best_hs]
batch_sizes = [32, 64, 128]
third_sweep_training_results = {}
third_sweep_configs = [
    (lr, bs, hs)
    for lr, bs, hs in product(lr_list, batch_sizes, hidden_sizes)
]
for lr, bs, hs in tqdm(third_sweep_configs, desc="Sweep"):
    config = copy.deepcopy(base_config)
    
    config["training"]["lr"] = lr
    config["training"]["batch_size"] = bs
    config["training"]["num_epochs"] = 100
    config["model"]["hidden_size"] = hs
    
    exp_name = f"lr{lr}_bs{bs}_hs{hs}"
    model = lstm.LSTMBaseline(**config["model"]).to(device)
    weights, train_arr, test_arr = train.fit_lstm(
        model, exp_name, train_dataset, test_dataset, **config["training"]
    )
    third_sweep_training_results[exp_name] = {
        "hyperparams": {"lr": lr, "bs": bs, "hs": hs},
        "weights": weights,
        "train": train_arr,
        "test": test_arr,
        "best_test": min(test_arr)
    }
    print()
best_third_exp = min(third_sweep_training_results, key=lambda k: third_sweep_training_results[k]["best_test"])
best_bs  = third_sweep_training_results[best_third_exp]["hyperparams"]["bs"]
print(f"Best config: {best_third_exp}, bs={best_bs}")

Sweep:   0%|          | 0/3 [00:00<?, ?it/s]

|lr0.00025_bs32_hs256| train = 0.0187, test= 0.0390
update LR: 0.00025 -> 0.000125
|lr0.00025_bs32_hs256| train = 0.0127, test= 0.0280
|lr0.00025_bs32_hs256| train = 0.0113, test= 0.0206
|lr0.00025_bs32_hs256| train = 0.0113, test= 0.0190
update LR: 0.000125 -> 6.25e-05
update LR: 6.25e-05 -> 3.125e-05
|lr0.00025_bs32_hs256| train = 0.0105, test= 0.0180
update LR: 3.125e-05 -> 1.5625e-05
update LR: 1.5625e-05 -> 7.8125e-06
| experiment: lr0.00025_bs32_hs256 | epoch 55, train: MSE 0.0103, test MSE: 0.0181
Stopping early

|lr0.00025_bs64_hs256| train = 0.0162, test= 0.0552
|lr0.00025_bs64_hs256| train = 0.0138, test= 0.0364
update LR: 0.00025 -> 0.000125
|lr0.00025_bs64_hs256| train = 0.0112, test= 0.0265
|lr0.00025_bs64_hs256| train = 0.0112, test= 0.0229
update LR: 0.000125 -> 6.25e-05
|lr0.00025_bs64_hs256| train = 0.0107, test= 0.0195
|lr0.00025_bs64_hs256| train = 0.0106, test= 0.0185
update LR: 6.25e-05 -> 3.125e-05
update LR: 3.125e-05 -> 1.5625e-05
update LR: 1.5625e-05 -> 7.8125

### Training on 80% of the nominal dataset (800 000 samples), validating on 200 000 samples.

In [8]:
full_training_dataset = LSTM_Dataset(device, 100, start = 0, end = 1000000 - 200000)
full_testing_dataset = LSTM_Dataset(device, 100, start = 1000000 - 200000, end = 1000000)

In [9]:
final_results = dict()
final_config = copy.deepcopy(base_config)

final_config["training"]["lr"] = best_lr
final_config["training"]["batch_size"] = best_bs
final_config["training"]["num_epochs"] = 100
final_config["model"]["hidden_size"] = best_hs

final_config["experiment_name"] = f"lr{best_lr}_bs{best_bs}_hs{best_hs}"
final_long_window_lstm = lstm.LSTMBaseline(**final_config["model"]).to(device)
final_weights, final_train_arr, final_test_arr = \
train.fit_lstm(final_long_window_lstm, final_config["experiment_name"], 
               full_training_dataset, full_testing_dataset, **final_config["training"], chunks = 80)

|lr0.00025_bs32_hs256| train = 0.0136, test= 0.0158
update LR: 0.00025 -> 0.000125
|lr0.00025_bs32_hs256| train = 0.0106, test= 0.0115
update LR: 0.000125 -> 6.25e-05
update LR: 6.25e-05 -> 3.125e-05
|lr0.00025_bs32_hs256| train = 0.0099, test= 0.0103
update LR: 3.125e-05 -> 1.5625e-05
update LR: 1.5625e-05 -> 7.8125e-06
update LR: 7.8125e-06 -> 3.90625e-06
|lr0.00025_bs32_hs256| train = 0.0098, test= 0.0101
update LR: 3.90625e-06 -> 1.953125e-06
update LR: 1.953125e-06 -> 9.765625e-07
update LR: 9.765625e-07 -> 4.8828125e-07
| experiment: lr0.00025_bs32_hs256 | epoch 48, train: MSE 0.0098, test MSE: 0.0101
Stopping early


In [10]:
final_long_window_lstm.load_state_dict(final_weights)
torch.save(final_long_window_lstm.state_dict(), "LSTM_long_window_weights.pth")